# Diabetes Risk Platform — Training Notebook

Runs the full 8-model comparison for both pipelines (screening + clinical).
Built to run on Kaggle, since this is too heavy for a laptop. The rest of
the project (ETL, feature engineering, scoring, dashboard) lives in the
`src/` and `app/` folders of the main repo, not here — this notebook is
intentionally self-contained (see note below) rather than importing from
`src/`, so it's a single portable file.

**Note on duplication:** the ETL/feature-engineering functions below
mirror `src/etl/*.py` and `src/features/*.py` in the main repo. That's a
deliberate exception, not an oversight — Kaggle notebooks need to be
single-file portable, so importing a local package isn't practical here.
If you change feature engineering, update both places.

## Setup on Kaggle
1. Add Data → search **`alexteboul/diabetes-health-indicators-dataset`** (screening)
2. Add Data → search **`iammustafatz/diabetes-prediction-dataset`** (clinical)
3. Run all cells
4. Download `/kaggle/working/outputs/` from the Output tab and drop it into
   the main repo's `outputs/` folder

## Setup

In [ ]:
# scikit-learn is pinned to match requirements/app.txt exactly — a version
# mismatch between training and the app is not just a warning, it can hard
# crash joblib.load() on newer scikit-learn releases (confirmed: a
# ColumnTransformer pickled with 1.6.1 fails to load under 1.8.0 with
# AttributeError: Can't get attribute '_RemainderColsList'). If you bump
# this, bump requirements/app.txt to match.
!pip -q install "scikit-learn==1.6.1" lightgbm catboost interpret imbalanced-learn shap --upgrade

In [ ]:
import os
import glob
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import shap

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from interpret.glassbox import ExplainableBoostingClassifier

import tensorflow as tf

In [ ]:
RANDOM_STATE = 42
TEST_SIZE = 0.2
TIERS = {"low": 0.30, "medium": 0.60}
# Models eligible for deployment. shap.TreeExplainer (used for fast, exact
# single-patient SHAP in the live dashboard) only supports tree ensembles —
# so if EBM/LR/NN/StackedEnsemble score highest overall, they're still
# reported below but the best TREE model is what gets deployed.
TREE_MODELS = {"XGBoost", "LightGBM", "CatBoost", "RandomForest"}

## Locate input files
Works whether run on Kaggle (`/kaggle/input/...`) or with files placed
locally under `data/*/raw/` for a quick local test of this same code.

In [ ]:
def find_input(filename, local_fallback):
    matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if matches:
        return matches[0]
    # Local dev: Jupyter's cwd is the notebook's own folder, so also try one
    # level up (repo root) regardless of where this was launched from.
    for candidate in (local_fallback, os.path.join("..", local_fallback)):
        if os.path.exists(candidate):
            return candidate
    raise FileNotFoundError(
        f"Could not find {filename}. On Kaggle, attach the dataset via "
        f"'Add Data'. Locally, place it at {local_fallback} (repo root)."
    )

SCREENING_PATH = find_input(
    "diabetes_binary_health_indicators_BRFSS2015.csv",
    "data/screening/raw/diabetes_binary_health_indicators_BRFSS2015.csv",
)
CLINICAL_PATH = find_input(
    "diabetes_prediction_dataset.csv",
    "data/clinical/raw/diabetes_prediction_dataset.csv",
)
print(SCREENING_PATH)
print(CLINICAL_PATH)

## Shared helpers — preprocessing, evaluation, the 8-model roster

In [ ]:
def build_preprocessor(continuous_cols, categorical_cols):
    """Scales continuous columns, one-hot encodes categorical columns, passes
    the rest through. Fit once on the training set so a single-row transform
    at inference time (in the dashboard) behaves identically to a full batch."""
    return ColumnTransformer(
        [("scale", StandardScaler(), continuous_cols),
         ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore",
                                   sparse_output=False), categorical_cols)],
        remainder="passthrough",
    )


def clean_feature_names(names):
    """Strips ColumnTransformer's 'step__' prefix so names stay human-readable."""
    return [n.split("__", 1)[1] if "__" in n else n for n in names]


def evaluate(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
    }


def best_threshold(y_true, y_prob, metric="f1"):
    best_t, best_score = 0.5, -1
    for t in np.arange(0.10, 0.90, 0.01):
        s = evaluate(y_true, y_prob, t)[metric]
        if s > best_score:
            best_t, best_score = t, s
    return round(float(best_t), 2)


def compute_sample_weight(y):
    """Uniform imbalance handling across every model, instead of five
    different library-specific class_weight APIs."""
    pos_weight = (y == 0).sum() / (y == 1).sum()
    return np.where(y == 1, pos_weight, 1.0)


def fit_model(model, X, y, sample_weight):
    try:
        model.fit(X, y, sample_weight=sample_weight)
    except TypeError:
        model.fit(X, y)
    return model


def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=str)

In [ ]:
class KerasBinaryClassifier:
    """Thin adapter so a Keras model drops into the same fit/predict_proba
    loop as every sklearn-style model below."""

    def __init__(self, input_dim, random_state=RANDOM_STATE):
        tf.random.set_seed(random_state)
        self.model = tf.keras.Sequential([
            tf.keras.layers.Input(shape=(input_dim,)),
            tf.keras.layers.Dense(32, activation="relu"),
            tf.keras.layers.Dropout(0.2),
            tf.keras.layers.Dense(16, activation="relu"),
            tf.keras.layers.Dense(1, activation="sigmoid"),
        ])
        self.model.compile(optimizer="adam", loss="binary_crossentropy")

    def fit(self, X, y, sample_weight=None):
        self.model.fit(X, y, epochs=15, batch_size=512, verbose=0, sample_weight=sample_weight)
        return self

    def predict_proba(self, X):
        p = self.model.predict(X, verbose=0).ravel()
        return np.column_stack([1 - p, p])

In [ ]:
def build_models(input_dim):
    return {
        "LogisticRegression": LogisticRegression(max_iter=1000),
        "RandomForest": RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
        "XGBoost": XGBClassifier(random_state=RANDOM_STATE, eval_metric="logloss"),
        "LightGBM": LGBMClassifier(random_state=RANDOM_STATE, verbosity=-1),
        "CatBoost": CatBoostClassifier(random_state=RANDOM_STATE, verbose=False),
        "NeuralNet": KerasBinaryClassifier(input_dim=input_dim),
        "EBM": ExplainableBoostingClassifier(random_state=RANDOM_STATE),
        "StackedEnsemble": StackingClassifier(
            estimators=[
                ("lr", LogisticRegression(max_iter=1000)),
                ("rf", RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)),
                ("xgb", XGBClassifier(random_state=RANDOM_STATE, eval_metric="logloss")),
            ],
            final_estimator=LogisticRegression(max_iter=1000), cv=5, n_jobs=-1,
        ),
    }


def compare_models(X_train, y_train, X_test, y_test):
    sw = compute_sample_weight(y_train)
    rows, fitted = [], {}
    for name, model in build_models(X_train.shape[1]).items():
        fit_model(model, X_train, y_train, sw)
        y_prob = model.predict_proba(X_test)[:, 1]
        t = best_threshold(y_test, y_prob)
        m = evaluate(y_test, y_prob, t)
        m.update(model=name, threshold=t)
        rows.append(m)
        fitted[name] = (model, y_prob, t)
        print(f"{name:18s} roc_auc={m['roc_auc']:.3f}  f1={m['f1']:.3f}  recall={m['recall']:.3f}")
    results = pd.DataFrame(rows).sort_values("roc_auc", ascending=False).reset_index(drop=True)
    return results, fitted

## The pipeline runner

One function, applied to both datasets — this is deliberate: the earlier
version of this project had the same feature-engineering logic copy-pasted
across a notebook, a training script, and a scoring script, and they'd
already drifted from each other. Applying one tested function to two
datasets, rather than writing it twice, is the fix.

In [ ]:
def run_pipeline(name, raw_path, load_and_clean_fn, engineer_fn, target_col,
                  continuous_cols, categorical_cols, output_dir):
    print(f"\n{'='*20} {name.upper()} {'='*20}")
    os.makedirs(os.path.join(output_dir, "models"), exist_ok=True)

    df = load_and_clean_fn(raw_path)
    X, y = df.drop(columns=[target_col]), df[target_col]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
    )
    X_train, X_test = engineer_fn(X_train), engineer_fn(X_test)

    preprocessor = build_preprocessor(continuous_cols, categorical_cols)
    X_train_s = preprocessor.fit_transform(X_train)
    X_test_s = preprocessor.transform(X_test)
    feature_names = clean_feature_names(preprocessor.get_feature_names_out())
    # Named DataFrames, not bare arrays: EBM's explain_global() falls back to
    # generic "feature_0001" names on a plain ndarray, which would silently
    # break the EBM-vs-SHAP comparison below.
    X_train_s = pd.DataFrame(X_train_s, columns=feature_names)
    X_test_s = pd.DataFrame(X_test_s, columns=feature_names)

    results, fitted = compare_models(X_train_s, y_train, X_test_s, y_test)
    print()
    print(results[["model", "accuracy", "precision", "recall", "f1", "roc_auc", "threshold"]].to_string(index=False))

    # Deployment pick: best ROC-AUC among tree models only (see TREE_MODELS note above)
    tree_results = results[results["model"].isin(TREE_MODELS)].reset_index(drop=True)
    deployed_name = tree_results.iloc[0]["model"]
    deployed_model, deployed_prob, deployed_threshold = fitted[deployed_name]
    overall_best = results.iloc[0]["model"]
    if overall_best != deployed_name:
        print(f"\nNote: {overall_best} scored highest overall, but {deployed_name} is "
              f"deployed — it's tree-based and supports fast SHAP explanations for live scoring.")
    print(f"Deployed model: {deployed_name}  (roc_auc={tree_results.iloc[0]['roc_auc']:.3f})")

    # SHAP export always comes from the deployed tree model
    explainer = shap.TreeExplainer(deployed_model)
    shap_values = explainer.shap_values(X_test_s)
    if isinstance(shap_values, list):
        shap_values = shap_values[1]
    expected_value = explainer.expected_value
    if not np.isscalar(expected_value):
        expected_value = expected_value[1]

    # EBM vs SHAP agreement — an inherently-interpretable model's own feature
    # rankings, compared against SHAP explanations of the deployed model
    ebm_global = fitted["EBM"][0].explain_global().data()
    ebm_importance = pd.Series(ebm_global["scores"], index=ebm_global["names"]).sort_values(ascending=False)
    shap_importance = pd.Series(np.abs(shap_values).mean(axis=0), index=feature_names).sort_values(ascending=False)
    print("\nTop 5 by EBM native importance:", list(ebm_importance.index[:5]))
    print(f"Top 5 by SHAP mean |value| ({deployed_name}):", list(shap_importance.index[:5]))

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    ebm_importance.head(10).iloc[::-1].plot.barh(ax=axes[0], title="EBM native importance")
    shap_importance.head(10).iloc[::-1].plot.barh(ax=axes[1], title=f"SHAP mean |value| ({deployed_name})")
    plt.tight_layout()
    plt.show()

    joblib.dump(preprocessor, os.path.join(output_dir, "models", "preprocessor.pkl"))
    joblib.dump(deployed_model, os.path.join(output_dir, "models", "model.pkl"))
    save_json(feature_names, os.path.join(output_dir, "models", "feature_names.json"))
    save_json({
        "model_name": deployed_name,
        "chosen_threshold": deployed_threshold,
        "test_metrics": {k: tree_results.iloc[0][k] for k in ["accuracy", "precision", "recall", "f1", "roc_auc"]},
        "random_state": RANDOM_STATE,
        "risk_tiers": TIERS,
        "overall_best_model": overall_best,
        "comparison_table": results.to_dict(orient="records"),
    }, os.path.join(output_dir, "models", "metadata.json"))

    agent_df = pd.DataFrame(shap_values, columns=feature_names)
    agent_df.insert(0, "predicted_prob", deployed_prob)
    agent_df.insert(1, "predicted_class", (deployed_prob >= deployed_threshold).astype(int))
    agent_df.insert(2, "true_class", y_test.reset_index(drop=True))
    agent_df["shap_base_value"] = expected_value
    agent_df.to_csv(os.path.join(output_dir, "shap_values_for_agent.csv"), index=False)

    print(f"\nArtifacts written to {output_dir}")
    return results

# Part 1 — Screening pipeline (BRFSS)

In [ ]:
SCREENING_BINARY_COLS = [
    "Diabetes_binary", "HighBP", "HighChol", "CholCheck", "Smoker", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost", "DiffWalk", "Sex",
]
SCREENING_ORDINAL_COLS = ["GenHlth", "MentHlth", "PhysHlth", "Age", "Education", "Income"]
SCREENING_CONTINUOUS = ["BMI", "GenHlth", "MentHlth", "PhysHlth", "Age", "Education",
                         "Income", "Health_Risk_Score", "Lifestyle_Score"]
SCREENING_CATEGORICAL = ["BMI_Category"]


def load_and_clean_screening(path):
    df = pd.read_csv(path)
    before = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Dropped {before - len(df):,} duplicate rows -> {len(df):,} remain")
    df[SCREENING_BINARY_COLS] = df[SCREENING_BINARY_COLS].astype(int)
    df[SCREENING_ORDINAL_COLS] = df[SCREENING_ORDINAL_COLS].astype(int)
    return df


def _bmi_category(bmi):
    if bmi < 18.5: return "Underweight"
    if bmi < 25: return "Normal"
    if bmi < 30: return "Overweight"
    return "Obese"


def engineer_features_screening(X):
    X = X.copy()
    X["Health_Risk_Score"] = (X["HighBP"] + X["HighChol"] + X["HeartDiseaseorAttack"]
                               + X["Stroke"] + (X["BMI"] >= 30).astype(int))
    X["Lifestyle_Score"] = X["PhysActivity"] + X["Fruits"] + X["Veggies"] - X["Smoker"]
    X["BMI_Category"] = X["BMI"].apply(_bmi_category)
    return X

### Quick EDA — target balance and correlation with target
(kept brief; the earlier project notebook already has a fuller EDA pass —
this is just a sanity check before training.)

In [ ]:
_df_preview = load_and_clean_screening(SCREENING_PATH)
print(_df_preview["Diabetes_binary"].value_counts(normalize=True))
_df_preview.corr(numeric_only=True)["Diabetes_binary"].drop("Diabetes_binary").sort_values(ascending=False).plot.barh(
    figsize=(6, 8), title="Correlation with target"
)
plt.tight_layout()
plt.show()
del _df_preview

In [ ]:
screening_results = run_pipeline(
    name="screening",
    raw_path=SCREENING_PATH,
    load_and_clean_fn=load_and_clean_screening,
    engineer_fn=engineer_features_screening,
    target_col="Diabetes_binary",
    continuous_cols=SCREENING_CONTINUOUS,
    categorical_cols=SCREENING_CATEGORICAL,
    output_dir="/kaggle/working/outputs/screening",
)

# Part 2 — Clinical pipeline (lab-based)

In [ ]:
CLINICAL_CONTINUOUS = ["age", "bmi", "HbA1c_level", "blood_glucose_level"]
CLINICAL_CATEGORICAL = ["gender", "smoking_history", "HbA1c_Category", "Glucose_Category"]


def load_and_clean_clinical(path):
    df = pd.read_csv(path)
    before = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Dropped {before - len(df):,} duplicate rows -> {len(df):,} remain")
    df["hypertension"] = df["hypertension"].astype(int)
    df["heart_disease"] = df["heart_disease"].astype(int)
    df["diabetes"] = df["diabetes"].astype(int)
    return df


def _hba1c_category(v):
    if v < 5.7: return "Normal"
    if v < 6.5: return "Prediabetes"
    return "Diabetes_range"


def _glucose_category(v):
    if v < 100: return "Normal"
    if v < 126: return "Prediabetes"
    return "Diabetes_range"


def engineer_features_clinical(X):
    X = X.copy()
    X["Comorbidity_Score"] = X["hypertension"] + X["heart_disease"]
    X["HbA1c_Category"] = X["HbA1c_level"].apply(_hba1c_category)
    X["Glucose_Category"] = X["blood_glucose_level"].apply(_glucose_category)
    return X

In [ ]:
_dfc_preview = load_and_clean_clinical(CLINICAL_PATH)
print(_dfc_preview["diabetes"].value_counts(normalize=True))
del _dfc_preview

In [ ]:
clinical_results = run_pipeline(
    name="clinical",
    raw_path=CLINICAL_PATH,
    load_and_clean_fn=load_and_clean_clinical,
    engineer_fn=engineer_features_clinical,
    target_col="diabetes",
    continuous_cols=CLINICAL_CONTINUOUS,
    categorical_cols=CLINICAL_CATEGORICAL,
    output_dir="/kaggle/working/outputs/clinical",
)

## Done

`/kaggle/working/outputs/` now has the same structure as the main repo's
`outputs/` folder:

```
outputs/screening/models/{preprocessor.pkl, model.pkl, feature_names.json, metadata.json}
outputs/screening/shap_values_for_agent.csv
outputs/clinical/models/{...}
outputs/clinical/shap_values_for_agent.csv
```

Download the `outputs/` folder from the Output tab and drop it into the
main repo, replacing the placeholder `outputs/` there. The dashboard
(`app/app.py`) picks it up automatically.

`metadata.json`'s `comparison_table` field has all 8 models' metrics — that
table is the actual "ML Models Comparison" deliverable, not just the one
deployed model.